# CUDA autodiff: inverse design of a 3D waveguide bend

Optimize a dielectric pattern that turns light from an input waveguide into the **fundamental guided mode** of a perpendicular output waveguide. `Simulation.differentiable()` supplies CUDA reverse-mode FDTD; JAX differentiates the density map and a fixed modal projection as well.

This adapts the geometry of the existing [3D topology-bend notebook](https://github.com/beamzorg/beamz/blob/38479de47d240b1b36018e724e2ace14ad521b56/examples/notebooks/topology_bend_3d.ipynb): a 5 × 5 × 2.2 µm air-clad domain, a 3 × 3 µm design square, a 200 nm thick dielectric core with relative permittivity 4, nominal 500 nm guides, and wavelength 1 µm. Training uses a 100 nm mesh and 200 fs window to keep GPU runtime manageable. Verification resamples the **fixed physical layout**, without refiltering or reoptimizing, at finer resolutions and longer windows.

The first version of this tutorial used net output flux and ten Adam updates. That was inadequate: radiation crossing the port inflated the score, and the unconstrained layout became asymmetric. This version uses outgoing **mode power**, 160 updates, projection continuation (β = 2, 4, 8, 16), and an explicit x/y transpose constraint. Symmetry is imposed as a design choice; reciprocity alone does not require an optimized layout to be symmetric.

The source and output planes lie 0.6 µm from their adjacent edges, outside CPML. A fixed straight-guide reference supplies the denominator. We check the complete gradient, CUDA/JAX agreement, ordinary mode analysis, binary thresholding, mesh/window sensitivity, reverse excitation, and field plots. Scores are ratios to the corresponding straight-guide mode power; they are not source-normalized absolute efficiencies or a claim of fabrication readiness.

The audit also uncovered an input/output inconsistency in the existing mode solver: Yee refinement was limited to x-directed ports. Uniform y-directed ports now use the same validated solve in a reflected x-normal frame, including the axial-vector sign of H. The reverse-excitation check below exercises that correction.

## Setup

Use a Python kernel with this BeamZ checkout, JAX, Optax, NumPy, and Matplotlib installed. For CUDA, build the optional component against a compatible JAX version, as described in [cuda/README.md](../../cuda/README.md); this API requires CUDA ABI 27 from this checkout. Native differentiation uses float32, one GPU, and 3D Yee/CPML physics.

The default `auto` policy selects CUDA when the compatible component is available and otherwise runs the same example with JAX. Set `BEAMZ_NOTEBOOK_BACKEND=cuda` before starting the kernel to require CUDA, or `jax` for the portable path. The notebook prints its actual backend. `BEAMZ_DOCS_TEST=1` selects the small CPU smoke configuration used by the example-notebook checks.

Executed notebooks and figures can be saved under `results/cuda_autodiff_waveguide_bend/`; the versioned source follows the repository convention of storing no cached outputs.

In [ ]:
import os
import sys
import time
from dataclasses import replace
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Rectangle as PlotRectangle

try:
    from IPython.display import Markdown, display
except ImportError:
    display = print
    Markdown = str

# Works from either the repository root or examples/notebooks; installed BeamZ
# also works when no checkout is found in the current directory's parents.
root = next(
    (
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "beamz/simulation/differentiation.py").is_file()
    ),
    Path.cwd(),
)
if (root / "beamz").is_dir():
    sys.path.insert(0, str(root))

import beamz as bz  # noqa: E402 -- checkout path must be established first
from beamz.design import MaterialGrid  # noqa: E402
from beamz.optimization.autodiff import transform_density  # noqa: E402
from beamz.optimization.modes import build_mode_projector  # noqa: E402

test_mode = os.environ.get("BEAMZ_DOCS_TEST") == "1"
backend = "jax" if test_mode else os.environ.get("BEAMZ_NOTEBOOK_BACKEND", "auto")
um = 1e-6
dx = 200e-9 if test_mode else 100e-9
run_time = 100e-15 if test_mode else 200e-15
updates = 2 if test_mode else 160
stage_length = 40
checkpoint_interval = 16 if test_mode else 32
beta, learning_rate = 2.0, 0.1
frequency = bz.LIGHT_SPEED / um
dt = 0.8 * dx / (bz.LIGHT_SPEED * np.sqrt(3.0))
times = np.arange(0.0, run_time, dt, dtype=np.float64)
artifacts = root / "results/cuda_autodiff_waveguide_bend"
if test_mode:
    artifacts = artifacts / "smoke"
artifacts.mkdir(parents=True, exist_ok=True)
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
print(
    f"JAX {jax.__version__} | policy={backend} | dx={dx / 1e-9:g} nm | "
    f"window={run_time / 1e-15:g} fs | {len(times)} timesteps"
)

## 1. Map a symmetric planar density into a 3D material grid

Train `(y, x)` logits and average them with their transpose **before** sigmoid, conic filtering, and projection. This enforces reflection across x = y, which exchanges the two fixed arms. Mask the input before convolution so inactive parameters cannot change the filtered edges. The entire constraint remains in the autodiff graph.

Only the design square changes. Extrusion gives a 200 nm core, resolved by two training cells. Cell masks retain their exact rasterized footprint during finer-grid verification, including the half-cell placement of a nominal 500 nm guide on the 100 nm mesh. All arrays use `(z, y, x)` storage and metre units.

In [ ]:
domain = np.array([5.0, 5.0, 2.2]) * um  # public x, y, z sizes
shape = tuple(int(n) for n in np.rint(domain[::-1] / dx))
# Integer bounds avoid floating-point inclusion of a cell on only one edge.
iz, iy, ix = np.indices(shape)
design_lo, design_hi = int(round(1.0 * um / dx)), int(round(4.0 * um / dx))
slab_lo, slab_hi = int(round(1.0 * um / dx)), int(round(1.2 * um / dx))
guide_cells = int(np.floor(0.5 * um / dx + 0.5 + 1e-9))
guide_lo = int(np.floor(2.5 * um / dx - guide_cells / 2 + 1e-9))
guide_hi = guide_lo + guide_cells
slab = (iz >= slab_lo) & (iz < slab_hi)
design_mask_np = (
    (ix[0] >= design_lo)
    & (ix[0] < design_hi)
    & (iy[0] >= design_lo)
    & (iy[0] < design_hi)
)
fixed_guides_np = ((ix[0] < design_lo) & (iy[0] >= guide_lo) & (iy[0] < guide_hi)) | (
    (iy[0] < design_lo) & (ix[0] >= guide_lo) & (ix[0] < guide_hi)
)
design_mask = jnp.asarray(design_mask_np)
fixed_guides = jnp.asarray(fixed_guides_np)
slab_mask = jnp.asarray(slab)
filter_radius = max(1, int(np.ceil(0.15 * um / dx)))
initial_parameters = jnp.zeros(shape[1:], dtype=jnp.float32)


def physical_density(parameters, projection_beta=beta):
    parameters = 0.5 * (parameters + parameters.T)
    return transform_density(
        jnp.where(design_mask, jax.nn.sigmoid(parameters), 0.0),
        design_mask,
        beta=projection_beta,
        eta=0.5,
        radius=filter_radius,
        filter_type="conic",
        fixed_structure_mask=fixed_guides,
    )


def permittivity(parameters, projection_beta=beta):
    pattern = jnp.where(
        design_mask,
        1.0 + 3.0 * physical_density(parameters, projection_beta),
        jnp.where(fixed_guides, 4.0, 1.0),
    )
    return jnp.where(slab_mask, pattern[None, :, :], 1.0).astype(jnp.float32)


initial_epsilon = permittivity(initial_parameters)
material_grid = MaterialGrid(
    permittivity=np.asarray(initial_epsilon),
    conductivity=np.float32(0.0),
    permeability=np.float32(1.0),
    resolution=dx,
    shape=shape,
)
print(
    f"Material cells (z, y, x): {shape} | trainable pixels: {design_mask_np.sum()} | "
    f"thickness: {np.count_nonzero(slab[:, 0, 0])} cells"
)
np.testing.assert_array_equal(design_mask_np, design_mask_np.T)
np.testing.assert_array_equal(fixed_guides_np, fixed_guides_np.T)

## 2. Compile the source, mode monitor, and differentiable solver

Inject a TE-like mode along +x. The output travels along −y; the modal amplitude at direction index 1 measures that outgoing branch. A `ModeMonitor` records all six field components on a common plane. Source profiles and the output-mode basis are solved once outside differentiation. Their local materials are fixed throughout optimization.

In [ ]:
source = bz.ModeSource(
    center=(0.6 * um, 2.5 * um, 1.1 * um),
    size=(0.0, 1.4 * um, 1.4 * um),
    direction="+",
    mode_spec=bz.ModeSpec(num_modes=1, polarization="te"),
    source_time=bz.GaussianPulse(
        freq0=frequency, fwidth=frequency / 12, offset=5 / (2 * np.pi)
    ),
)
output_monitor = bz.ModeMonitor(
    center=(2.5 * um, 0.6 * um, 1.1 * um),
    size=(1.4 * um, 0.0, 1.4 * um),
    freqs=[frequency],
    name="output",
    mode_spec=bz.ModeSpec(num_modes=1, polarization="te"),
)
boundaries = [bz.PML(thickness=0.3 * um, formulation="cpml")]
simulation = bz.Simulation(
    material_grid=material_grid,
    time=times,
    sources=[source],
    monitors=[output_monitor],
    boundaries=boundaries,
    normalize_source=None,
)
solver = simulation.differentiable(
    backend=backend, checkpoint_interval=checkpoint_interval
)
resolved_backend = solver.program.config.backend
print(f"Actual solver: {resolved_backend} | checkpoint interval: {checkpoint_interval}")

## 3. Freeze a guided-mode objective and straight-guide reference

A straight guide uses the exact same rasterized transverse core, source, pulse, mesh, and time window. Its reference plane has the same nominal source-to-port path length as the bend. `build_mode_projector` freezes the **same coupled forward/backward mode basis, normalization, and interpolation** used by `SimulationResults.mode()`.

The projector accepts the common-time phasors from `solver.monitor_fields()` and returns `(frequency, direction, mode_index)` amplitudes, with direction indices 0 = +axis and 1 = −axis. Differentiate `|a_out|² / P_reference`; radiation orthogonal to that fixed guided mode cannot improve this objective. The denominator never changes during training. Ordinary forward runs validate the projection before optimization.

In [ ]:
# Copy the rasterized input cross-section; do not rerasterize nominal bounds.
straight_epsilon = np.repeat(np.asarray(initial_epsilon)[:, :, :1], shape[2], axis=2)
reference_grid = replace(material_grid, permittivity=straight_epsilon)
reference_monitor = bz.ModeMonitor(
    center=(4.4 * um, 2.5 * um, 1.1 * um),
    size=(0.0, 1.4 * um, 1.4 * um),
    freqs=[frequency],
    mode_spec=bz.ModeSpec(num_modes=1, polarization="te"),
    name="reference",
)
reference_simulation = simulation.updated_copy(
    material_grid=reference_grid, monitors=[reference_monitor]
)
reference_result = reference_simulation.run(backend=backend, performance=False)
reference_power = jax.lax.stop_gradient(
    jnp.asarray(
        abs(reference_result.mode("reference").amps.sel(direction="+").values[0, 0])
        ** 2,
        dtype=jnp.float32,
    )
)
reference_flux = float(np.asarray(reference_result["reference"].flux)[0])
assert np.isfinite(float(reference_power)) and float(reference_power) > 0
initial_forward_result = simulation.run(backend=backend, performance=False)
project_output = build_mode_projector(initial_forward_result, "output")
initial_state = solver()
np.testing.assert_allclose(
    np.asarray(project_output(solver.monitor_fields(initial_state, "output"))),
    initial_forward_result.mode("output").amps.values,
    rtol=0.003,
    atol=1e-6,
)


def transmission(parameters, projection_beta=beta):
    state = solver(permittivity(parameters, projection_beta))
    amplitude = project_output(solver.monitor_fields(state, "output"))[0, 1, 0]
    return jnp.abs(amplitude) ** 2 / reference_power


def loss(parameters, projection_beta=beta):
    return -transmission(parameters, projection_beta)


evaluate = jax.jit(jax.value_and_grad(loss))
score = jax.jit(transmission)
initial_loss, initial_gradient = evaluate(initial_parameters)
initial_transmission = float(-initial_loss)
print(f"Initial guided-mode / straight guide: {100 * initial_transmission:.3f}%")
print(f"Complete density-gradient norm: {float(jnp.linalg.norm(initial_gradient)):.6g}")
np.testing.assert_allclose(initial_gradient, initial_gradient.T, rtol=1e-5, atol=1e-7)

### Inspect the initial material and signed gradient

The gradient includes the symmetry map, sigmoid, filtering, projection, extrusion, FDTD, and modal measurement. A negative loss gradient means increasing that logit improves guided-mode power. The dashed square marks the trainable region. The parameter gradient must obey the same diagonal symmetry as the material.

In [ ]:
mid_z = int(np.argmin(np.abs((np.arange(shape[0]) + 0.5) * dx - 1.1 * um)))
extent = [0.0, shape[2] * dx / um, 0.0, shape[1] * dx / um]
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
epsilon_image = axes[0].imshow(
    np.asarray(initial_epsilon)[mid_z],
    origin="lower",
    extent=extent,
    cmap="cividis",
    vmin=1.0,
    vmax=4.0,
)
grad_limit = float(jnp.max(jnp.abs(initial_gradient)))
gradient_image = axes[1].imshow(
    np.asarray(initial_gradient),
    origin="lower",
    extent=extent,
    cmap="PuOr",
    vmin=-grad_limit,
    vmax=grad_limit,
)
for ax in axes:
    ax.add_patch(
        PlotRectangle(
            (1, 1), 3, 3, fill=False, edgecolor="#202020", linestyle="--", linewidth=1.2
        )
    )
    ax.set(xlabel="x (µm)", ylabel="y (µm)")
axes[0].set_title("Initial relative permittivity · z = 1.1 µm")
axes[1].set_title("Signed loss gradient with respect to logits")
fig.colorbar(epsilon_image, ax=axes[0], label="Relative permittivity")
fig.colorbar(gradient_image, ax=axes[1], label="∂loss / ∂logit")
fig.savefig(artifacts / "initial_material_and_gradient.png", dpi=160)
plt.show()

### Check the full chain with finite differences

Probe the normalized gradient direction. The finite-difference calculation evaluates only the scalar forward objective; it does not use the backward kernel. Several perturbation sizes expose cancellation in float32 arithmetic. Also check that parameters outside the design mask have exactly zero gradient.

At the optimized point we repeat the check with larger perturbations and compare the full CUDA gradient with JAX again. A step-size sweep is necessary when finite differences of a nearly stationary float32 objective suffer cancellation.

In [ ]:
gradient_norm = jnp.linalg.norm(initial_gradient)
assert np.isfinite(float(gradient_norm)) and float(gradient_norm) > 0
np.testing.assert_array_equal(np.asarray(initial_gradient)[~design_mask_np], 0.0)
direction = initial_gradient / gradient_norm
autodiff_derivative = float(jnp.vdot(initial_gradient, direction))
gradient_checks = []
for h in (0.02, 0.01, 0.005):
    finite_difference = float(
        (
            loss(initial_parameters + h * direction)
            - loss(initial_parameters - h * direction)
        )
        / (2 * h)
    )
    error = abs(finite_difference - autodiff_derivative) / abs(autodiff_derivative)
    gradient_checks.append((h, autodiff_derivative, finite_difference, error))
    assert error < 0.02, f"Gradient mismatch at h={h}: {error:.2%}"
table = "| Step h | Autodiff | Finite difference | Relative error |\n|---:|---:|---:|---:|\n"
table += "\n".join(
    f"| {h:g} | {ad:.7g} | {fd:.7g} | {err:.3%} |" for h, ad, fd, err in gradient_checks
)
display(Markdown(table))

### Compare CUDA with the JAX reference when CUDA is active

Use the same materials, source profiles, monitor definition, fixed denominator, and checkpoint interval. This compares complete parameter gradients as well as the objective. On a JAX-only run, the finite-difference check above remains the independent derivative check; this cell explicitly reports that no CUDA comparison was made.

In [ ]:
backend_gradient_error = None
if resolved_backend != "jax":
    jax_solver = simulation.differentiable(
        backend="jax", checkpoint_interval=checkpoint_interval
    )

    def jax_loss(parameters, projection_beta):
        state = jax_solver(permittivity(parameters, projection_beta))
        amplitude = project_output(jax_solver.monitor_fields(state, "output"))[0, 1, 0]
        return -(jnp.abs(amplitude) ** 2) / reference_power

    jax_evaluate = jax.jit(jax.value_and_grad(jax_loss))
    jax_value, jax_gradient = jax_evaluate(initial_parameters, jnp.float32(beta))
    backend_gradient_error = float(
        jnp.linalg.norm(initial_gradient - jax_gradient) / jnp.linalg.norm(jax_gradient)
    )
    np.testing.assert_allclose(
        float(initial_loss), float(jax_value), rtol=5e-4, atol=1e-6
    )
    assert backend_gradient_error < 0.005
    print(
        f"CUDA/JAX complete-gradient relative difference: {backend_gradient_error:.4%}"
    )
else:
    print("JAX execution: finite differences passed; CUDA/JAX comparison not run.")

## 4. Optimize with Adam and projection continuation

Run 40 updates at each β = 2, 4, 8, 16. Sharpening gradually avoids forcing a binary layout before a useful optical path exists. The complete scalar objective has no handwritten adjoint source or field-overlap gradient.

Each history entry is evaluated on the design at its labeled update count; the last entry is recomputed **after the last update**. Changing β changes the material at a stage boundary, so the plotted history is not one fixed-β convergence curve. Symmetry is enforced throughout, rather than applied to an already optimized asymmetric result. Save and verify the hard-thresholded layout separately.

In [ ]:
optimizer = optax.adam(learning_rate)
parameters = initial_parameters
optimizer_state = optimizer.init(parameters)


@jax.jit
def optimization_step(parameters, optimizer_state, projection_beta):
    objective_value, gradient = evaluate(parameters, projection_beta)
    changes, optimizer_state = optimizer.update(gradient, optimizer_state, parameters)
    parameters = optax.apply_updates(parameters, changes)
    return parameters, optimizer_state, objective_value, jnp.linalg.norm(gradient)


history, gradient_history, beta_history = [], [], []
started = time.perf_counter()
for iteration in range(updates):
    current_beta = float(2 ** (1 + min(iteration // stage_length, 3)))
    parameters, optimizer_state, old_loss, norm = optimization_step(
        parameters, optimizer_state, jnp.float32(current_beta)
    )
    # old_loss describes the layout at exactly `iteration` completed updates.
    history.append(float(-old_loss))
    gradient_history.append(float(norm))
    beta_history.append(current_beta)
    if iteration % 10 == 0 or test_mode:
        print(
            f"Design at update {iteration:3d}: mode / straight guide = "
            f"{100 * history[-1]:7.3f}% | beta={current_beta:g}",
            flush=True,
        )
final_beta = jnp.float32(current_beta)
final_transmission = float(score(parameters, final_beta))
history.append(final_transmission)
beta_history.append(current_beta)
elapsed = time.perf_counter() - started
final_epsilon = permittivity(parameters, final_beta)
binary_epsilon = np.where(np.asarray(final_epsilon) >= 2.5, 4.0, 1.0).astype(np.float32)
print(f"Final updated design: {100 * final_transmission:.3f}% | {elapsed:.1f} s")
assert np.all(np.isfinite(history)) and final_transmission > initial_transmission
np.testing.assert_allclose(history[0], initial_transmission, rtol=1e-5)
np.testing.assert_array_equal(
    np.asarray(final_epsilon)[~np.asarray(slab_mask & design_mask[None])],
    np.asarray(initial_epsilon)[~np.asarray(slab_mask & design_mask[None])],
)
symmetry_error = float(
    np.linalg.norm(np.asarray(final_epsilon) - np.asarray(final_epsilon).swapaxes(1, 2))
    / np.linalg.norm(np.asarray(final_epsilon))
)
assert symmetry_error < 1e-6
np.testing.assert_array_equal(binary_epsilon, binary_epsilon.swapaxes(1, 2))
# Check the complete gradient again at the optimized, sharply projected design.
final_loss, final_gradient = evaluate(parameters, final_beta)
final_direction = final_gradient / jnp.linalg.norm(final_gradient)
# At a nearly stationary float32 objective, a single tiny perturbation can
# suffer cancellation. Check a step-size sweep and require both larger steps
# to agree; retain every measurement in the saved artifact.
final_ad = float(jnp.vdot(final_gradient, final_direction))
final_gradient_checks = []
for h in (0.2, 0.1, 0.05):
    final_fd = float(
        (
            loss(parameters + h * final_direction, final_beta)
            - loss(parameters - h * final_direction, final_beta)
        )
        / (2 * h)
    )
    error = abs(final_fd - final_ad) / abs(final_ad)
    final_gradient_checks.append((h, final_ad, final_fd, error))
    print(
        f"Final gradient, h={h:g}: autodiff={final_ad:.7g}, "
        f"finite difference={final_fd:.7g}, relative error={error:.3%}"
    )
final_gradient_error = max(row[3] for row in final_gradient_checks[:2])
assert final_gradient_error < 0.02
final_backend_gradient_error = None
if resolved_backend != "jax":
    final_jax_value, final_jax_gradient = jax_evaluate(parameters, final_beta)
    final_backend_gradient_error = float(
        jnp.linalg.norm(final_gradient - final_jax_gradient)
        / jnp.linalg.norm(final_jax_gradient)
    )
    np.testing.assert_allclose(
        float(final_loss), float(final_jax_value), rtol=5e-4, atol=1e-6
    )
    assert final_backend_gradient_error < 0.005
    print(f"Final CUDA/JAX gradient difference: {final_backend_gradient_error:.4%}")
print(f"Material transpose asymmetry: {symmetry_error:.3g}")
np.savez_compressed(
    artifacts / "optimization.npz",
    initial_parameters=np.asarray(initial_parameters),
    final_parameters=np.asarray(parameters),
    initial_epsilon=np.asarray(initial_epsilon),
    final_epsilon=np.asarray(final_epsilon),
    binary_epsilon=binary_epsilon,
    transmission=np.asarray(history),
    gradient_norm=np.asarray(gradient_history),
    beta_history=np.asarray(beta_history),
    gradient_checks=np.asarray(gradient_checks),
    final_gradient_error=final_gradient_error,
    final_gradient_checks=np.asarray(final_gradient_checks),
    dx=dx,
    dt=dt,
    run_time=run_time,
    reference_flux=float(reference_flux),
    reference_power=float(reference_power),
    symmetry_error=symmetry_error,
)

### Compare initial, continuous, and binary designs

All material panels share the range [1, 4]. The thresholded panel is an independently verified material, rather than a replacement for the continuous score. Stage boundaries mark the β changes. Training history alone does not establish finer-mesh performance.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8), layout="constrained")
for ax, epsilon, title in zip(
    axes[:3],
    (initial_epsilon, final_epsilon, binary_epsilon),
    ("Initial", "Optimized continuous", "Thresholded binary"),
    strict=True,
):
    image = ax.imshow(
        np.asarray(epsilon)[mid_z],
        origin="lower",
        extent=extent,
        cmap="cividis",
        vmin=1.0,
        vmax=4.0,
    )
    ax.add_patch(
        PlotRectangle(
            (1, 1), 3, 3, fill=False, edgecolor="#202020", linestyle="--", linewidth=1.2
        )
    )
    ax.set(title=title, xlabel="x (µm)", ylabel="y (µm)")
fig.colorbar(image, ax=axes[:3], label="Relative permittivity", shrink=0.88)
axes[3].plot(
    np.arange(updates + 1),
    100 * np.asarray(history),
    color="#2864A8",
    marker="o",
    markersize=4,
)
axes[3].set(
    title="Objective after each update",
    xlabel="Adam update",
    ylabel="Guided mode / straight guide (%)",
    ylim=(0, None),
)
axes[3].set_xticks(np.unique(np.linspace(0, updates, min(6, updates + 1), dtype=int)))
axes[3].grid(axis="y", alpha=0.2)
for boundary in range(stage_length, updates, stage_length):
    axes[3].axvline(boundary, color="0.5", linestyle=":", linewidth=1)
fig.savefig(artifacts / "optimization_summary.png", dpi=160)
plt.show()
display(
    Markdown(
        f"**Observed result:** guided-mode power / straight guide changed from "
        f"**{100 * initial_transmission:.3f}%** to **{100 * final_transmission:.3f}%** "
        f"after **{updates} updates** using **{resolved_backend}**. "
        f"Optimization-loop time, including its first compilation: **{elapsed:.1f} s**."
    )
)

## 5. Check ordinary forward runs and inspect fields

Run the initial, optimized continuous, and binary materials with the standard execution path. Ordinary `result.mode("output")` must reproduce the differentiable guided-mode score. Report **net flux separately** to expose radiation in the old metric. The field panels show frequency-domain |E|² at 1 µm with the same source amplitude and color scale.

In [ ]:
field_monitor = bz.FieldMonitor(
    center=(2.5 * um, 2.5 * um, 1.1 * um),
    size=(4.4 * um, 4.4 * um, 0.0),
    freqs=[frequency],
    fields=("Ex", "Ey", "Ez"),
    name="fields",
)
inspection_results, forward_scores, net_flux_scores = [], [], []
for epsilon in (initial_epsilon, final_epsilon, binary_epsilon):
    inspected = simulation.updated_copy(
        material_grid=replace(material_grid, permittivity=np.asarray(epsilon)),
        monitors=[output_monitor, field_monitor],
    )
    result = inspected.run(backend=backend, performance=False)
    inspection_results.append(result)
    forward_scores.append(
        float(abs(result.mode("output").amps.sel(direction="-").values[0, 0]) ** 2)
        / float(reference_power)
    )
    net_flux_scores.append(
        -float(np.asarray(result["output"].flux)[0]) / reference_flux
    )
np.testing.assert_allclose(
    forward_scores[:2],
    [initial_transmission, final_transmission],
    rtol=0.003,
    atol=1e-6,
)
binary_state = solver(jnp.asarray(binary_epsilon))
binary_score = float(
    jnp.abs(project_output(solver.monitor_fields(binary_state, "output"))[0, 1, 0]) ** 2
    / reference_power
)
np.testing.assert_allclose(forward_scores[2], binary_score, rtol=0.003, atol=1e-6)
for name, modal, net in zip(
    ("Initial", "Continuous", "Binary"), forward_scores, net_flux_scores, strict=True
):
    print(f"{name}: guided mode={100 * modal:.3f}% | net flux={100 * net:.3f}%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.2))
fig.suptitle("DFT |E|² at λ = 1 µm")
field_images = []
for ax, result, title in zip(
    axes,
    inspection_results,
    ("Initial", "Optimized continuous", "Thresholded binary"),
    strict=True,
):
    result.plot_field(
        "fields",
        field="E",
        frequency=frequency,
        val="abs^2",
        ax=ax,
        cmap="magma",
        colorbar=False,
        show=False,
    )
    ax.set_title(title)
    field_images.append(
        next(
            artist
            for artist in ax.get_children()
            if isinstance(artist, ScalarMappable) and artist.get_array() is not None
        )
    )
field_max = max(float(np.max(image.get_array())) for image in field_images)
for image in field_images:
    image.set_clim(0.0, field_max)
fig.colorbar(field_images[-1], ax=axes, label="DFT |E|² (shared scale)", shrink=0.88)
fig.savefig(
    artifacts / "initial_and_optimized_fields.png", dpi=160, bbox_inches="tight"
)
plt.show()

## 6. Verify the fixed layout on finer meshes, longer windows, and reversed excitation

Resample the piecewise-constant material with exact integer cell subdivision. Do **not** apply the density filter again: that would change the physical device. Rebuild the source and port mode at each resolution, and use the same resampled transverse core in that resolution's straight-guide reference. The 400 fs check isolates finite-window sensitivity; 50 nm and 25 nm checks assess spatial sensitivity. Reverse excitation swaps the two arms and normalizes against a separately solved straight guide.

The table contains actual forward results for continuous and binary layouts. A transpose-symmetric reciprocal layout should give comparable forward/reverse modal transmission. Finite-grid discrepancies and mesh changes remain visible; scores are never clipped. `BEAMZ_BEND_FINEST=0` skips the optional 25 nm check for a shorter GPU run. Reduced CPU smoke mode skips these expensive checks.

In [ ]:
verification_rows = []
for layout, modal, net in zip(
    ("Continuous", "Binary"), forward_scores[1:], net_flux_scores[1:], strict=True
):
    verification_rows.append(
        (layout, dx / 1e-9, run_time / 1e-15, "Forward", modal, net)
    )


def subdivide(epsilon, factor):
    for axis in range(3):
        epsilon = np.repeat(epsilon, factor, axis=axis)
    return epsilon


def verify_fixed_layouts(factor, duration, reverse=False):
    fine_dx = dx / factor
    fine_dt = 0.8 * fine_dx / (bz.LIGHT_SPEED * np.sqrt(3))
    fine_time = np.arange(0.0, duration, fine_dt)
    fine_reference_epsilon = subdivide(straight_epsilon, factor)
    fine_source, fine_output, fine_reference = source, output_monitor, reference_monitor
    if reverse:
        fine_reference_epsilon = fine_reference_epsilon.swapaxes(1, 2)
        fine_source = replace(
            source, center=output_monitor.center, size=output_monitor.size
        )
        fine_output = replace(output_monitor, center=source.center, size=source.size)
        fine_reference = replace(
            reference_monitor,
            center=(2.5 * um, 4.4 * um, 1.1 * um),
            size=(1.4 * um, 0.0, 1.4 * um),
        )
    fine_grid = MaterialGrid(
        conductivity=np.float32(0.0),
        permeability=np.float32(1.0),
        permittivity=fine_reference_epsilon,
        resolution=fine_dx,
        shape=fine_reference_epsilon.shape,
    )
    fine_simulation = simulation.updated_copy(
        material_grid=fine_grid,
        time=fine_time,
        sources=[fine_source],
        monitors=[fine_reference],
    )
    reference = fine_simulation.run(backend=backend, performance=False)
    denominator = float(
        abs(reference.mode("reference").amps.sel(direction="+").values[0, 0]) ** 2
    )
    flux_denominator = float(np.asarray(reference["reference"].flux)[0])
    assert denominator > 0 and np.isfinite(denominator)
    for layout, epsilon in (("Continuous", final_epsilon), ("Binary", binary_epsilon)):
        epsilon = subdivide(np.asarray(epsilon), factor)
        result = fine_simulation.updated_copy(
            material_grid=replace(fine_grid, permittivity=epsilon),
            monitors=[fine_output],
        ).run(backend=backend, performance=False)
        power = float(
            abs(result.mode("output").amps.sel(direction="-").values[0, 0]) ** 2
        )
        net_flux = -float(np.asarray(result["output"].flux)[0]) / flux_denominator
        row = (
            layout,
            fine_dx / 1e-9,
            duration / 1e-15,
            "Reverse" if reverse else "Forward",
            power / denominator,
            net_flux,
        )
        verification_rows.append(row)
        print(
            f"{layout}, {row[1]:g} nm, {row[2]:g} fs, {row[3]}: "
            f"mode / straight guide = {100 * row[4]:.3f}%",
            flush=True,
        )


if not test_mode:
    verify_fixed_layouts(1, 400e-15)
    verify_fixed_layouts(2, 400e-15)
    verify_fixed_layouts(2, 400e-15, reverse=True)
    if os.environ.get("BEAMZ_BEND_FINEST", "1") != "0":
        verify_fixed_layouts(4, 400e-15)
else:
    print("Reduced CPU smoke run: mesh/window/reverse verification skipped.")
if not test_mode:
    for layout in ("Continuous", "Binary"):
        forward = next(
            row[4]
            for row in verification_rows
            if row[0] == layout
            and np.isclose(row[1], 50.0)
            and np.isclose(row[2], 400.0)
            and row[3] == "Forward"
        )
        reverse = next(
            row[4]
            for row in verification_rows
            if row[0] == layout
            and np.isclose(row[1], 50.0)
            and np.isclose(row[2], 400.0)
            and row[3] == "Reverse"
        )
        np.testing.assert_allclose(forward, reverse, rtol=0.001, atol=1e-5)
header = (
    "| Layout | Mesh (nm) | Window (fs) | Direction | Guided mode (%) | Net flux (%) |\n"
    "|---|---:|---:|---|---:|---:|\n"
)
header += "\n".join(
    f"| {name} | {mesh:g} | {window:g} | {direction} | "
    f"{100 * modal:.3f} | {100 * net:.3f} |"
    for name, mesh, window, direction, modal, net in verification_rows
)
display(Markdown(header))
import json

(artifacts / "verification.json").write_text(
    json.dumps(
        [
            dict(
                layout=row[0],
                dx_nm=row[1],
                window_fs=row[2],
                direction=row[3],
                modal_transmission=row[4],
                net_flux_ratio=row[5],
            )
            for row in verification_rows
        ],
        indent=2,
    )
)

## Interpreting the result

Use the **guided-mode** column to assess this bend. Net flux includes unwanted radiation. Compare continuous and binary values before choosing a layout, and compare mesh/window rows before claiming numerical convergence. The diagonal symmetry is an explicit constraint and is checked independently in the saved material arrays.

The core has only two training cells through its thickness. Even good transmission here does not validate sidewall roughness, loss, bandwidth, minimum feature size, or fabrication robustness. For those tasks, optimize on a finer grid, include multiple frequencies and erosion/dilation objectives, and verify on another refined grid. A fixed checkpoint interval stores full boundary states and replays intervening steps during reverse mode.

CUDA currently provides first-order reverse mode on one GPU with float32 Yee/CPML state; it excludes BF16 CPML state, sharding, coupled full-tensor updates, JVPs, and higher derivatives. See the [optimization guide](../../beamz/optimization/README.md). Artifacts contain the executed notebook, figures, saved continuous/binary materials and histories, gradient checks, and verification table.